# unialg Explorer

A gentle tour of the current `unialg` API.

A **morphism** is a typed arrow. You can run it, compose it, branch it, add parameters, add effects, and lift it through reusable data shapes.

The notebook keeps backend setup in `explore_support.py` so the main path can stay focused on the algebraic API.

## Setup

Run this once before the examples. It loads the local package and imports a small set of demo morphisms.

In [1]:
import sys
_VENV = "/home/scanbot/unialg/.venv/lib/python3.12/site-packages"
_SRC = "/home/scanbot/unialg/src"
for _p in (_VENV, _SRC):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from unialg import (
    identity, copy, delete, fst, snd, inl, inr,
    compose, par, pair, case,
    assoc, symmetry,
    Id, One, Const, Sum, Prod,
    Functor, Optic, Semiring,
    act, act_forward, act_backward, compose_optic,
    cata, ana, hylo,
    lower, apply_poly, poly_fmap
)
# from src.unialg.semantics.functors import apply_poly
# from src.unialg.structure.actions import poly_fmap
from explore_support import *

print("Ready.")

ImportError: cannot import name 'act' from 'unialg' (/home/scanbot/unialg/src/unialg/__init__.py)

## 1. Plain Morphisms

`add1` and `double` are ordinary morphisms from `Int` to `Int`. The helper `show_morphism` prints the logical type, and `run_int` runs a morphism on an integer input.

In [ ]:
print("add1: ", show_morphism(add1))
print("double:", show_morphism(double))

print("add1(5)  =", run_int(add1, 5))
print("double(5) =", run_int(double, 5))

add1:  Int -> Int
double: Int -> Int
add1(5)  = 6
double(5) = 10


## 2. Composition

`compose(f, g)` means: run `f`, then run `g`. The output type of `f` must match the input type of `g`.

In [ ]:
add_then_double = compose(add1, double)

print("add_then_double:", show_morphism(add_then_double))
print("add_then_double(3) =", run_int(add_then_double, 3))

add_then_double: param (<function ProductType at 0x7f1644339c60>, <hydra.core.TypeUnit object at 0x7f1644145f40>), input Int -> Int


RuntimeError: Reduction failed for Morphism(node=Compose(f=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.add')), argument=TermVariable(value=Name(value='x')))), argument=TermLiteral(value=LiteralInteger(value=IntegerValueInt32(value=1))))))), dom=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)), cod=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))), g=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.mul')), argument=TermVariable(value=Name(value='x')))), argument=TermLiteral(value=LiteralInteger(value=IntegerValueInt32(value=2))))))), dom=TypeLiteral(value=LiteralTypeInteger(value=<Intege

In [ ]:
try:
    compose(add1, add_pair)
except Exception as e:
    print(type(e).__name__ + ": add1 returns Int, but add_pair expects Int x Int")

MorphismError: add1 returns Int, but add_pair expects Int x Int


## 3. Products

Products let us talk about pairs.

`copy` duplicates one input. `fst` and `snd` project from a pair. `pair(f, g)` sends the same input to both morphisms. `par(f, g)` sends the left input to `f` and the right input to `g`.

In [ ]:
copy_int = copy(INT)
fst_int = fst(INT_PAIR)
snd_int = snd(INT_PAIR)

print("copy:", show_morphism(copy_int), "   copy(7) =", run_pair(copy_int, 7))
print("fst: ", show_morphism(fst_int), "   fst(3, 9) =", run_int_from_pair(fst_int, (3, 9)))
print("snd: ", show_morphism(snd_int), "   snd(3, 9) =", run_int_from_pair(snd_int, (3, 9)))

copy: Int -> Int x Int    copy(7) = (7, 7)
fst:  Int x Int -> Int    fst(3, 9) = 3
snd:  Int x Int -> Int    snd(3, 9) = 9


In [ ]:
same_input = pair(add1, double)
side_by_side = par(add1, double)

print("pair(add1, double):", show_morphism(same_input))
print("pair(add1, double)(5) =", run_pair(same_input, 5))

print("par(add1, double): ", show_morphism(side_by_side))
print("par(add1, double)(5, 6) =", run_pair(side_by_side, (5, 6)))

pair(add1, double): param (<function ProductType at 0x7f1644339c60>, <hydra.core.TypeUnit object at 0x7f1644145f40>), input Int -> Int x Int


RuntimeError: Reduction failed for Morphism(node=Pair(f=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.add')), argument=TermVariable(value=Name(value='x')))), argument=TermLiteral(value=LiteralInteger(value=IntegerValueInt32(value=1))))))), dom=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)), cod=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))), g=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.mul')), argument=TermVariable(value=Name(value='x')))), argument=TermLiteral(value=LiteralInteger(value=IntegerValueInt32(value=2))))))), dom=TypeLiteral(value=LiteralTypeInteger(value=<IntegerTy

## 4. Sums

Sums represent a choice: a value arrives from the left side or the right side.

`case(f, g)` branches on that choice. Left values go to `f`; right values go to `g`.

In [ ]:
branch = case(add1, double)

print("branch:", show_morphism(branch))
print("branch(Left 5)  =", run_left_int(branch, 5))
print("branch(Right 5) =", run_right_int(branch, 5))

## 5. Parameters

A parametric morphism has an extra environment value. In this example, the parameter is an integer that gets added to the input.

The important surface idea is simple: provide a parameter and an input. The support helper owns the runtime packing.

In [ ]:
print("add_param:", show_morphism(add_param))
print("add_param(param=10, value=3) =", run_para_int(add_param, param=10, value=3))

In [ ]:
two_params = compose(add_param, add_param_again)

print("compose(add_param, add_param_again):", show_morphism(two_params))
print("with f_param=10, g_param=20, value=3:",
      run_composed_para_int(two_params, f_param=10, g_param=20, value=3))

## 6. Effects

A lax morphism returns its result inside an effect.

Here `Maybe` means the result is wrapped as an optional value, and `List` means the morphism can return multiple results. Composition sequences those effects automatically.

In [ ]:
maybe_pipeline = compose(maybe_add1, maybe_double)

print("maybe_pipeline:", show_morphism(maybe_pipeline))
print("maybe_pipeline(3) =", run_maybe_int(maybe_pipeline, 3))

In [ ]:
list_pipeline = compose(list_step, list_double)

print("list_pipeline:", show_morphism(list_pipeline))
print("list_pipeline(3) =", run_list_int(list_pipeline, 3))

## 7. Shapes / Polynomial Functors

A polynomial functor is a reusable data shape. `Id()` marks the places where values of the current type live.

`poly_fmap(shape, h)` lifts a morphism through the shape, applying `h` at every `Id()` position.

In [ ]:
pair_shape = Prod(Id(), Id())

print("shape:", pair_shape.pretty())
print("shape applied to Int:", show_type(apply_poly(pair_shape, INT)))

In [ ]:
lifted = poly_fmap(Functor("Test", pair_shape), add1)

print("lifted add1:", show_morphism(lifted))
print("lifted add1 on (3, 7) =", run_pair(lifted, (3, 7)))

In [ ]:
lifted_param = poly_fmap(Functor("Test", pair_shape), add_param)

print("lifted add_param:", show_morphism(lifted_param))
print("param=10, value=(3, 7):", run_para_pair(lifted_param, param=10, value=(3, 7)))

In [ ]:
lifted_maybe = poly_fmap(Functor("Test", pair_shape), maybe_add1)

print("lifted maybe_add1:", show_morphism(lifted_maybe))
print("lifted maybe_add1 on (3, 7) =", run_maybe_pair(lifted_maybe, (3, 7)))

In [ ]:
sum_shape = Sum(Id(), Id())
lifted_sum = poly_fmap(Functor("Test", sum_shape), add1)

print("sum shape:", sum_shape.pretty())
print("Left 5  ->", run_sum_int(lifted_sum, side="left", value=5))
print("Right 5 ->", run_sum_int(lifted_sum, side="right", value=5))

## 8. Structural Morphisms — assoc and symmetry

`assoc` and `symmetry` are canonical structural morphisms built from types alone — no primitives needed.

`assoc((A×B)×C)` gives the reassociation `(A×B)×C → A×(B×C)`.  
`symmetry(A×B)` gives the swap `A×B → B×A`.

In [ ]:
assoc_m = assoc(INT_TRIPLE_L)
print("assoc:", show_morphism(assoc_m))
print("assoc((1,2),3) =", run_assoc(assoc_m, 1, 2, 3))

sym_m = symmetry(INT_PAIR)
print("symmetry:", show_morphism(sym_m))
print("symmetry(3,7) =", run_pair(sym_m, (3, 7)))

## 8. Lowering

The examples above use friendly runners. Underneath, `lower(morphism)` compiles the morphism expression into a raw Hydra term. `run(morphism, argument, ctx, graph)` applies that term and reduces it.

You usually do not need to look at the lowered term while using the API, but it is useful when checking the compiler boundary.

In [ ]:
compiled = lower(add_then_double)

print("expression:", add_then_double.node.pretty())
print("lowered term class:", type(compiled).__name__)

## 9. What Can Be Composed?

`compose(f, g)` accepts `Morphism` objects. It does not accept raw Python functions directly.

A Python function can still participate, but it first has to become a Hydra `Primitive`, then a `Morphism` can point at that primitive. That backend pattern belongs in fixture/support code, not in the main tutorial path.

In [ ]:
print("compose takes Morphism values:")
print("  add1 is", type(add1).__name__)
print("  lambda x: x + 1 is", type(lambda x: x + 1).__name__)

## 10. Appendix: Where Did The Fixtures Come From?

The sample morphisms and notebook-friendly runners live in `explore_support.py`.

That file contains the raw Hydra details: primitive names, `expr.Prim(...)` leaves, argument packing, result unwrapping, and the `ctx` / `graph` used by `run`. Keeping that code out of the main path makes this notebook about the `unialg` API rather than Hydra plumbing.

## 11. Try It Yourself

Some small experiments:

- Compose `double` then `add3` and run it on `5`.
- Build `pair(add1, add3)` and run it on `10`.
- Build `par(double, add3)` and run it on `(2, 8)`.
- Lift `double` through `pair_shape` with `poly_fmap`.
- Compose a plain morphism with a Maybe morphism and check the result.

In [ ]:
double_then_add3 = compose(double, add3)

print("double_then_add3:", show_morphism(double_then_add3))
print("double_then_add3(5) =", run_int(double_then_add3, 5))

In [ ]:
add3_then_double = compose(add3, double)
thendoubleagain = pair(add3_then_double, double)
print(run_pair(thendoubleagain, 21))

## 12. Optics

An `Optic` is a triple `(functor, forward, backward)` where the functor describes the container shape and the two morphisms decompose and reconstruct around it.

`act(optic, h)` applies `h` through the optic: decompose with `forward`, lift `h` through the functor with `poly_fmap`, then reconstruct with `backward`.

Here a **pair lens** focuses on the first element of `Int × Int` using the functor `F(X) = X × Int`.

In [ ]:
pair_lens = Optic(
    functor=Functor("PairFirst", Prod(Id(), Const(INT))),
    forward=identity(INT_PAIR),
    backward=identity(INT_PAIR),
)

print("pair_lens.source:", show_type(pair_lens.source))
print("pair_lens.target:", show_type(pair_lens.target))
print("pair_lens.focus: ", show_type(pair_lens.focus))

lifted = act(pair_lens, add1)
print("act(pair_lens, add1):", show_morphism(lifted))
print("act(pair_lens, add1)(3, 7) =", run_pair(lifted, (3, 7)))

pair_lens.source: Int x Int
pair_lens.target: Int x Int
pair_lens.focus:  Int
act(pair_lens, add1): param (<function ProductType at 0x7f1644339c60>, (<function ProductType at 0x7f1644339c60>, <hydra.core.TypeUnit object at 0x7f1644145f40>)), input Int x Int -> Int x Int


RuntimeError: Reduction failed for Morphism(node=Compose(f=Compose(f=Identity(space=TypePair(value=PairType(first=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)), second=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))))), g=Prim(raw=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.pairs.bimap')), argument=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.add')), argument=TermVariable(value=Name(value='x')))), argument=TermLiteral(value=LiteralInteger(value=IntegerValueInt32(value=1))))))))), argument=TermLambda(value=Lambda(parameter=Name(value='x_'), domain=Nothing, body=TermVariable(value=Name(value='x_')))))), dom=TypePair(value=PairType(first=TypeLiteral(value=LiteralTypeInteger(

## 13. Semiring

A `Semiring` captures the algebraic structure of two binary operations on a carrier type. All four components (`plus`, `times`, `zero`, `one`) are `Morphism` objects, so they participate in the typed composition machinery.

In [ ]:
int_semiring = Semiring(
    name="int-add-mul",
    carrier=INT,
    plus=add_pair,
    times=mul_pair,
    zero=const_zero,
    one=const_one,
)

print("name:    ", int_semiring.name)
print("carrier: ", show_type(int_semiring.carrier))
print("plus:    ", show_morphism(int_semiring.plus))
print("times:   ", show_morphism(int_semiring.times))
print("zero:    ", show_morphism(int_semiring.zero))
print("one:     ", show_morphism(int_semiring.one))

print("plus(3, 4) =", run_int_from_pair(int_semiring.plus, (3, 4)))
print("times(3,4) =", run_int_from_pair(int_semiring.times, (3, 4)))

name:     int-add-mul
carrier:  Int
plus:     Int x Int -> Int
times:    Int x Int -> Int
zero:     Unit -> Int
one:      Unit -> Int
plus(3, 4) = 7
times(3,4) = 12


In [ ]:
def check_commutative(op, name):
    # op ∘ symmetry == op
    swapped = compose(symmetry(INT_PAIR), op)

    for a, b in [(2, 3), (0, 5), (-2, 7)]:
        lhs = run_int_from_pair(op, (a, b))
        rhs = run_int_from_pair(swapped, (a, b))
        print(f"{name} comm ({a}, {b}):", lhs, "==", rhs, lhs == rhs)


def check_associative(op, name):
    # left:  ((A × A) × A) --(op × id)--> A × A --op--> A
    left = compose(par(op, identity(INT)), op)

    # right: ((A × A) × A) --assoc--> A × (A × A) --(id × op)--> A × A --op--> A
    right = compose(
        compose(assoc(INT_TRIPLE_L), par(identity(INT), op)),
        op,
    )

    for a, b, c in [(1, 2, 3), (0, 4, 5), (-2, 7, 3)]:
        arg = nested_left_arg(a, b, c)
        lhs = int_val(run(left, arg, ctx, graph))
        rhs = int_val(run(right, arg, ctx, graph))
        print(f"{name} assoc ({a}, {b}, {c}):", lhs, "==", rhs, lhs == rhs)

In [ ]:
check_commutative(int_semiring.plus, "plus")
check_associative(int_semiring.plus, "plus")

check_commutative(int_semiring.times, "times")
check_associative(int_semiring.times, "times")

RuntimeError: Reduction failed for Morphism(node=Compose(f=Symmetry(dom=TypePair(value=PairType(first=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)), second=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)))), cod=TypePair(value=PairType(first=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)), second=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))))), g=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='p'), domain=Nothing, body=TermApplication(value=Application(function=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.math.add')), argument=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.pairs.first')), argument=TermVariable(value=Name(value='p')))))), argument=TermApplication(value=Application(function=TermVariable(value=Name(value='hydra.lib.pairs.second')), argum

## 14. Recursive Schemes — cata, ana, hylo

Catamorphisms (`cata`), anamorphisms (`ana`), and hylomorphisms (`hylo`) are uniform recursion patterns over an `Optic` fixed point.

An `Optic` becomes a recursive carrier by supplying `forward = unroll` and `backward = roll` together with an explicit `carrier` type.

The demo below uses a trivially terminating carrier over `Int` with shape `F(X) = 1 + X`:
- `cata` folds: the algebra `1 + Int → Int` always returns 7
- `ana` unfolds: the coalgebra `Int → 1 + Int` always stops immediately, rolling to 42
- `hylo` chains them

In [ ]:
fp = one_or_self_optic(rolled_value=42)

stop_coalg = compose(delete(INT), inl(SumType(UNIT, INT)))
fold_alg   = case(const_int(7), identity(INT))

folded    = cata(fp, fold_alg)
unfolded  = ana(fp, stop_coalg)
transform = hylo(fp, stop_coalg, fold_alg)

print("cata: algebra folds any input to 7")
print("  cata(999) =", run_int(folded, 999))

print("ana:  coalgebra stops immediately, rolls to 42")
print("  ana(5) =", run_int(unfolded, 5))

print("hylo: unfold then fold → always 7")
print("  hylo(999) =", run_int(transform, 999))

AssertionError: Unreachable: all variants handled

## 15. Backend Morphisms

`BackendOps.from_spec` loads a JSON spec and registers each operation as a Hydra primitive with a canonical name (`unialg.backend.<op>`).  The same name is used regardless of which library backs it — swapping backends means changing the spec file, not the DSL.

`run` automatically includes `aux_primitives` from the morphism, so no manual graph extension is needed.

In [ ]:
from pathlib import Path
import math as _math

from unialg import run
from unialg.structure.backend import BackendOps
import hydra.dsl.meta.phantoms as P

_SPEC = Path("/home/scanbot/unialg/src/unialg/tensors/backends/jax.json")
ops = BackendOps.from_spec(_SPEC)

print("Loaded ops:", ", ".join(sorted(ops.keys())))
print()


def _fval(term):
    return term.value.value.value


def run_f2(m, a, b):
    """Run a binary float morphism on (a, b)."""
    return _fval(run(m, P.pair(P.float64(a), P.float64(b)).value, ctx, graph))


def run_f1(m, a):
    """Run a unary float morphism on a."""
    return _fval(run(m, P.float64(a).value, ctx, graph))


# --- elementwise binary ops ---
print("add(2, 3)          =", run_f2(ops["add"],       2.0,  3.0))
print("subtract(5, 1.5)   =", run_f2(ops["subtract"],  5.0,  1.5))
print("multiply(2, 3)     =", run_f2(ops["multiply"],  2.0,  3.0))
print("divide(9, 2)       =", run_f2(ops["divide"],    9.0,  2.0))
print("power(2, 10)       =", run_f2(ops["power"],     2.0, 10.0))
print("maximum(3, 7)      =", run_f2(ops["maximum"],   3.0,  7.0))
print("minimum(3, 7)      =", run_f2(ops["minimum"],   3.0,  7.0))
print("logaddexp(0, 0)    =", run_f2(ops["logaddexp"], 0.0,  0.0))
print()

# --- unary ops ---
print("exp(1)             =", run_f1(ops["exp"],       1.0))
print("log(e)             =", run_f1(ops["log"],   _math.e))
print("log1p(0)           =", run_f1(ops["log1p"],     0.0))
print("sqrt(16)           =", run_f1(ops["sqrt"],     16.0))
print("square(5)          =", run_f1(ops["square"],    5.0))
print("abs(-3.5)          =", run_f1(ops["abs"],      -3.5))
print("neg(4)             =", run_f1(ops["neg"],       4.0))
print("tanh(0)            =", run_f1(ops["tanh"],      0.0))
print("reciprocal(4)      =", run_f1(ops["reciprocal"],4.0))
print("sigmoid(0)         =", run_f1(ops["sigmoid"],   0.0))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Loaded ops: abs, add, cos, divide, exp, log, log1p, logaddexp, maximum, minimum, multiply, neg, power, reciprocal, reduce.add, reduce.logaddexp, reduce.maximum, reduce.minimum, reduce.multiply, relu, sigmoid, sign, sin, softmax, softplus, sqrt, square, subtract, tanh

add(2, 3)          = 5.0
subtract(5, 1.5)   = 3.5
multiply(2, 3)     = 6.0
divide(9, 2)       = 4.5
power(2, 10)       = 1024.0
maximum(3, 7)      = 7.0
minimum(3, 7)      = 3.0
logaddexp(0, 0)    = 0.6931471824645996

exp(1)             = 2.7182817459106445
log(e)             = 1.0
log1p(0)           = 0.0
sqrt(16)           = 4.0
square(5)          = 25.0
abs(-3.5)          = 3.5
neg(4)             = -4.0
tanh(0)            = 0.0
reciprocal(4)      = 0.25
sigmoid(0)         = 0.5


In [ ]:
# Minimal backend-neutral contraction IR + rewrite framework
# No NumPy execution. No hardcoded semiring implementation.
# Only symbolic contraction rewriting.

import hydra.core as core
import hydra.rewriting as rw
import hydra.dsl.meta.phantoms as P


# ------------------------------------------------------------------
# Basic Hydra term helpers
# ------------------------------------------------------------------

def hname(s):
    return core.Name(s)

def var(s):
    return core.TermVariable(hname(s))

def app(f, x):
    return core.TermApplication(core.Application(f, x))

def appn(f, *args):
    t = f
    for a in args:
        t = app(t, a)
    return t


# ------------------------------------------------------------------
# Structural inspection
# ------------------------------------------------------------------

def get_app_parts(t):
    if type(t).__name__ != "TermApplication":
        return None

    v = getattr(t, "value", None)
    if v is None:
        return None

    f = getattr(v, "function", None)
    x = getattr(v, "argument", None)

    if f is None or x is None:
        return None

    return f, x

def flatten_application(t):
    args = []
    cur = t

    while True:
        parts = get_app_parts(cur)
        if parts is None:
            return cur, list(reversed(args))

        f, arg = parts
        args.append(arg)
        cur = f


# ------------------------------------------------------------------
# Literal helpers
# ------------------------------------------------------------------

def wrap_string(s):
    return P.string(str(s)).value

def unwrap_string(t):
    while hasattr(t, "value"):
        t = t.value
    return t


# ------------------------------------------------------------------
# Rewrite framework
# ------------------------------------------------------------------

rewrite_term = rw.rewrite_term

def bottom_up(local_rule):
    def rule(recurse, t):
        return local_rule(recurse(t))
    return rule

def normalize_hydra(term, rule, max_rounds=20):
    cur = term

    for _ in range(max_rounds):
        prev = cur
        cur = rewrite_term(rule, cur)

        if cur == prev:
            return cur

    raise RuntimeError("rewrite did not converge")


# ------------------------------------------------------------------
# Symbolic contraction IR
# ------------------------------------------------------------------

CONTRACT = var("contract")

def contract(semiring, equation, *operands):
    """
    contract(sr, eq, A, B, ...)
    """
    return appn(
        CONTRACT,
        semiring,
        wrap_string(equation),
        *operands,
    )

def match_contract(t):
    head, args = flatten_application(t)

    if head != CONTRACT:
        return None

    if len(args) < 3:
        return None

    semiring = args[0]
    equation = unwrap_string(args[1])
    operands = args[2:]

    return semiring, equation, operands


# ------------------------------------------------------------------
# Equation parsing
# ------------------------------------------------------------------

def parse_equation(eq):
    lhs, rhs = eq.split("->", 1)
    return tuple(lhs.split(",")), rhs

def build_equation(inputs, output):
    return ",".join(inputs) + "->" + output


# ------------------------------------------------------------------
# Generic contraction fusion rewrite
# ------------------------------------------------------------------

def fuse_contract_once(t):
    outer = match_contract(t)
    if outer is None:
        return t

    outer_sr, outer_eq, outer_ops = outer
    outer_inputs, outer_output = parse_equation(outer_eq)

    for pos, op in enumerate(outer_ops):
        inner = match_contract(op)

        if inner is None:
            continue

        inner_sr, inner_eq, inner_ops = inner

        # semiring mismatch => cannot fuse
        if inner_sr != outer_sr:
            continue

        inner_inputs, inner_output = parse_equation(inner_eq)

        # inner output must feed the outer input slot
        if pos >= len(outer_inputs):
            continue

        if inner_output != outer_inputs[pos]:
            continue

        fused_inputs = (
            outer_inputs[:pos]
            + inner_inputs
            + outer_inputs[pos + 1:]
        )

        fused_ops = (
            tuple(outer_ops[:pos])
            + tuple(inner_ops)
            + tuple(outer_ops[pos + 1:])
        )

        fused_eq = build_equation(fused_inputs, outer_output)

        return contract(
            outer_sr,
            fused_eq,
            *fused_ops,
        )

    return t

fuse_contract_rule = bottom_up(fuse_contract_once)

In [ ]:
# Semiring-aware symbolic contraction IR
# Uses actual backend op morphism terms as opaque handles.
# No NumPy arrays. No execution. This only tests rewrite legality.

def semiring(plus_elem, times_elem, plus_reduce, times_reduce):
    return appn(
        var("semiring"),
        plus_elem,
        times_elem,
        plus_reduce,
        times_reduce,
    )

def contract(sr, equation, *operands):
    return appn(
        var("contract"),
        sr,
        wrap_string(equation),
        *operands,
    )

def match_contract(t):
    head, args = flatten_application(t)

    if head != var("contract"):
        return None

    if len(args) < 3:
        return None

    sr = args[0]
    eq = unwrap_string(args[1])
    operands = args[2:]

    return sr, eq, operands

def parse_equation(eq):
    lhs, rhs = eq.split("->", 1)
    return tuple(lhs.split(",")), rhs

def build_equation(inputs, output):
    return ",".join(inputs) + "->" + output

def fuse_contract_once(t):
    outer = match_contract(t)
    if outer is None:
        return t

    outer_sr, outer_eq, outer_ops = outer
    outer_inputs, outer_output = parse_equation(outer_eq)

    for pos, op in enumerate(outer_ops):
        inner = match_contract(op)
        if inner is None:
            continue

        inner_sr, inner_eq, inner_ops = inner

        # Important: fusion is only valid under the same semiring ops.
        if inner_sr != outer_sr:
            continue

        inner_inputs, inner_output = parse_equation(inner_eq)

        if pos >= len(outer_inputs):
            continue

        if inner_output != outer_inputs[pos]:
            continue

        fused_inputs = (
            outer_inputs[:pos]
            + inner_inputs
            + outer_inputs[pos + 1:]
        )

        fused_ops = (
            tuple(outer_ops[:pos])
            + tuple(inner_ops)
            + tuple(outer_ops[pos + 1:])
        )

        return contract(
            outer_sr,
            build_equation(fused_inputs, outer_output),
            *fused_ops,
        )

    return t

fuse_contract_rule = bottom_up(fuse_contract_once)

In [ ]:
# -------------------------
# Backend-op-backed semiring IR
# -------------------------

def op_key(name):
    """
    Symbolic reference to a backend op loaded by BackendOps.from_spec.
    Execution later resolves this through ops[name].
    """
    if name not in ops:
        raise KeyError(f"Backend op not loaded: {name}. Available: {sorted(ops.keys())}")
    return wrap_string(name)


def semiring(plus_elem, times_elem, plus_reduce, times_reduce):
    """
    Semiring is represented by four backend op keys:

      plus_elem     elementwise ⊕
      times_elem    elementwise ⊗
      plus_reduce   reduction ⊕
      times_reduce  reduction ⊗

    These are symbolic handles, not Morphism internals.
    """
    return appn(
        var("semiring"),
        op_key(plus_elem),
        op_key(times_elem),
        op_key(plus_reduce),
        op_key(times_reduce),
    )


def match_semiring(t):
    head, args = flatten_application(t)

    if head != var("semiring"):
        return None

    if len(args) != 4:
        return None

    return tuple(unwrap_string(a) for a in args)


def contract(sr, equation, *operands):
    return appn(
        var("contract"),
        sr,
        wrap_string(equation),
        *operands,
    )


def match_contract(t):
    head, args = flatten_application(t)

    if head != var("contract"):
        return None

    if len(args) < 3:
        return None

    sr = args[0]
    eq = unwrap_string(args[1])
    operands = args[2:]

    return sr, eq, operands

In [ ]:
sr_add_mul = semiring(
    plus_elem="add",
    times_elem="multiply",
    plus_reduce="reduce.add",
    times_reduce="reduce.multiply",
)

print(match_semiring(sr_add_mul))

('add', 'multiply', 'reduce.add', 'reduce.multiply')


In [ ]:
# -------------------------
# Generic contraction fusion using semiring equality
# -------------------------

def parse_equation(eq):
    lhs, rhs = eq.split("->", 1)
    return tuple(lhs.split(",")), rhs

def build_equation(inputs, output):
    return ",".join(inputs) + "->" + output

def fuse_contract_once(t):
    outer = match_contract(t)
    if outer is None:
        return t

    outer_sr, outer_eq, outer_ops = outer
    outer_inputs, outer_output = parse_equation(outer_eq)

    for pos, op in enumerate(outer_ops):
        inner = match_contract(op)
        if inner is None:
            continue

        inner_sr, inner_eq, inner_ops = inner

        # Only fuse if both contractions use the exact same semiring declaration
        if inner_sr != outer_sr:
            continue

        inner_inputs, inner_output = parse_equation(inner_eq)

        # The inner result must feed the corresponding outer input slot
        if pos >= len(outer_inputs) or inner_output != outer_inputs[pos]:
            continue

        fused_inputs = (
            outer_inputs[:pos]
            + inner_inputs
            + outer_inputs[pos + 1:]
        )

        fused_ops = (
            tuple(outer_ops[:pos])
            + tuple(inner_ops)
            + tuple(outer_ops[pos + 1:])
        )

        return contract(
            outer_sr,
            build_equation(fused_inputs, outer_output),
            *fused_ops,
        )

    return t

fuse_contract_rule = bottom_up(fuse_contract_once)

In [ ]:
# -------------------------
# Test: same semiring fuses
# -------------------------

A = var("A")
B = var("B")
C = var("C")

nested = contract(
    sr_add_mul,
    "ik,kl->il",
    contract(sr_add_mul, "ij,jk->ik", A, B),
    C,
)

fused = rewrite_term(fuse_contract_rule, nested)

sr, eq, operands = match_contract(fused)

print("equation:", eq)
print("semiring:", match_semiring(sr))
print("operands:", operands)

assert eq == "ij,jk,kl->il"
assert operands == [A, B, C]
assert sr == sr_add_mul

equation: ij,jk,kl->il
semiring: ('add', 'multiply', 'reduce.add', 'reduce.multiply')
operands: [TermVariable(value=Name(value='A')), TermVariable(value=Name(value='B')), TermVariable(value=Name(value='C'))]


In [ ]:
# -------------------------
# Test: different semiring does NOT fuse
# -------------------------

sr_max_mul = semiring(
    plus_elem="maximum",
    times_elem="multiply",
    plus_reduce="reduce.maximum",
    times_reduce="reduce.multiply",
)

nested_mismatch = contract(
    sr_add_mul,
    "ik,kl->il",
    contract(sr_max_mul, "ij,jk->ik", A, B),
    C,
)

not_fused = rewrite_term(fuse_contract_rule, nested_mismatch)

print(not_fused)

assert not_fused == nested_mismatch

In [ ]:
# Execute symbolic contract terms over real tensors using semiring op keys

import importlib
import json
from pathlib import Path
import numpy as np

_SPEC = Path("/home/scanbot/unialg/src/unialg/tensors/backends/jax.json")

with open(_SPEC) as f:
    _BACKEND_SPEC = json.load(f)

def _resolve_path(path):
    mod, attr = path.rsplit(".", 1)
    return getattr(importlib.import_module(mod), attr)

def _backend_fn(op_name):
    if op_name not in _BACKEND_SPEC["operations"]:
        raise KeyError(f"Unknown backend op: {op_name}")
    return _resolve_path(_BACKEND_SPEC["operations"][op_name]["path"])

def _eval_semiring(sr):
    plus_elem, times_elem, plus_reduce, times_reduce = match_semiring(sr)
    return {
        "plus_elem": _backend_fn(plus_elem),
        "times_elem": _backend_fn(times_elem),
        "plus_reduce": _backend_fn(plus_reduce),
        "times_reduce": _backend_fn(times_reduce),
    }

def _broadcast_for_equation(arr, labels, all_labels):
    shape = []
    arr_axis = 0

    for label in all_labels:
        if label in labels:
            shape.append(arr.shape[arr_axis])
            arr_axis += 1
        else:
            shape.append(1)

    return arr.reshape(shape)

def execute_contract_term(term, env):
    """
    Evaluate:
        contract(sr, "ij,jk->ik", A, B)

    env maps symbolic variable names to real tensors:
        {"A": array, "B": array}
    """
    matched = match_contract(term)

    if matched is None:
        # variable leaf
        if type(term).__name__ == "TermVariable":
            name = unwrap_string(term.value)
            return env[name]

        raise TypeError(f"Cannot execute non-contract term: {term}")

    sr, eq, operands = matched
    ops4 = _eval_semiring(sr)

    input_labels, output_labels = parse_equation(eq)

    values = [execute_contract_term(op, env) for op in operands]

    all_labels = tuple(dict.fromkeys("".join(input_labels)))
    reduce_axes = tuple(i for i, label in enumerate(all_labels) if label not in output_labels)

    factors = [
        _broadcast_for_equation(value, labels, all_labels)
        for value, labels in zip(values, input_labels)
    ]

    acc = factors[0]
    for factor in factors[1:]:
        acc = ops4["times_elem"](acc, factor)

    if reduce_axes:
        acc = ops4["plus_reduce"](acc, axis=reduce_axes)

    remaining = tuple(label for label in all_labels if label in output_labels)

    if remaining != tuple(output_labels):
        perm = [remaining.index(label) for label in output_labels]
        acc = np.asarray(acc).transpose(perm)

    return np.asarray(acc)


# Real tensor test

sr_add_mul = semiring(
    plus_elem="add",
    times_elem="multiply",
    plus_reduce="reduce.add",
    times_reduce="reduce.multiply",
)

A = var("A")
B = var("B")
C = var("C")

nested = contract(
    sr_add_mul,
    "ik,kl->il",
    contract(sr_add_mul, "ij,jk->ik", A, B),
    C,
)

fused = rewrite_term(fuse_contract_rule, nested)

i, j, k, l = 5, 3, 4, 2

env = {
    "A": np.random.rand(i, j),
    "B": np.random.rand(j, k),
    "C": np.random.rand(k, l),
}

nested_out = execute_contract_term(nested, env)
fused_out = execute_contract_term(fused, env)

print("nested shape:", nested_out.shape)
print("fused shape: ", fused_out.shape)
print("same:", np.allclose(nested_out, fused_out))

assert np.allclose(nested_out, fused_out)

In [ ]:
# Execute nested vs rewritten contraction over real tensors

A0 = var("A0")
B0 = var("B0")
C0 = var("C0")
D0 = var("D0")

nested_complex = contract(
    sr_add_mul,
    "abe,ef->abf",
    contract(
        sr_add_mul,
        "abd,bde->abe",
        contract(sr_add_mul, "abc,cd->abd", A0, B0),
        C0,
    ),
    D0,
)

fused_complex = normalize_hydra(nested_complex, fuse_contract_rule)

env_complex = {
    "A0": np.random.rand(2, 3, 4),  # abc
    "B0": np.random.rand(4, 5),     # cd
    "C0": np.random.rand(3, 5, 6),  # bde
    "D0": np.random.rand(6, 7),     # ef
}

nested_out = execute_contract_term(nested_complex, env_complex)
fused_out = execute_contract_term(fused_complex, env_complex)

print("nested_out:")
print(nested_out)

print("\nfused_out:")
print(fused_out)

print("\nshapes:")
print("nested:", nested_out.shape)
print("fused: ", fused_out.shape)

print("\nequal:", np.allclose(nested_out, fused_out))

assert nested_out.shape == (2, 3, 7)
assert fused_out.shape == (2, 3, 7)
assert np.allclose(nested_out, fused_out)

In [ ]:

### Cell 2 — Code

from __future__ import annotations

from dataclasses import dataclass, field
from collections.abc import Mapping, Callable
import numpy as np

# This section is designed to be runnable immediately after section 15.
# If it is run standalone, reload the backend operation table.
try:
    ops
except NameError:
    from pathlib import Path
    from unialg.structure.backend import BackendOps

    _SPEC = Path("/home/scanbot/unialg/src/unialg/tensors/backends/jax.json")
    ops = BackendOps.from_spec(_SPEC)

print("Backend op handles available:", ", ".join(sorted(ops.keys())))

In [ ]:
# ---------------------------------------------------------------------
# Equation syntax
# ---------------------------------------------------------------------

Labels = tuple[str, ...]

def labels(spec: str | Labels) -> Labels:
    """Single-character index labels for now: 'ij' -> ('i', 'j')."""
    if isinstance(spec, tuple):
        return spec
    return tuple(spec)

def show_labels(xs: Labels) -> str:
    return "".join(xs)

@dataclass(frozen=True)
class Equation:
    """A parsed contraction equation: inputs -> output."""
    inputs: tuple[Labels, ...]
    output: Labels

    @classmethod
    def parse(cls, text: str) -> "Equation":
        lhs, rhs = text.split("->", 1)
        ins = tuple(labels(part.strip()) for part in lhs.split(",") if part.strip())
        out = labels(rhs.strip())
        return cls(ins, out)

    def __str__(self) -> str:
        return ",".join(show_labels(i) for i in self.inputs) + "->" + show_labels(self.output)

    def replace_input(self, pos: int, new_inputs: tuple[Labels, ...]) -> "Equation":
        return Equation(
            self.inputs[:pos] + new_inputs + self.inputs[pos + 1:],
            self.output,
        )

In [ ]:
# ---------------------------------------------------------------------
# Tensor expression syntax
# ---------------------------------------------------------------------

class TensorExpr:
    @property
    def output_labels(self) -> Labels:
        raise NotImplementedError

@dataclass(frozen=True)
class TensorVar(TensorExpr):
    name: str
    labels: str | Labels

    def __post_init__(self):
        object.__setattr__(self, "labels", labels(self.labels))

    @property
    def output_labels(self) -> Labels:
        return self.labels

    def __repr__(self) -> str:
        return f"{self.name}:{show_labels(self.labels)}"

@dataclass(frozen=True)
class TensorSemiring:
    """
    Tensor-level semiring handle.

    The keys are stable symbolic names from the backend spec.
    The morphism fields come from BackendOps, so this wrapper still points
    back into the existing unialg backend/morphism layer.
    """
    name: str
    plus_elem_key: str
    times_elem_key: str
    plus_reduce_key: str
    times_reduce_key: str

    plus_elem: object = field(compare=False, repr=False)
    times_elem: object = field(compare=False, repr=False)
    plus_reduce: object = field(compare=False, repr=False)
    times_reduce: object = field(compare=False, repr=False)

    @classmethod
    def from_ops(
        cls,
        name: str,
        ops,
        *,
        plus_elem: str,
        times_elem: str,
        plus_reduce: str,
        times_reduce: str,
    ) -> "TensorSemiring":
        for key in (plus_elem, times_elem, plus_reduce, times_reduce):
            if key not in ops:
                raise KeyError(f"Backend op not loaded: {key}. Available: {sorted(ops.keys())}")

        return cls(
            name=name,
            plus_elem_key=plus_elem,
            times_elem_key=times_elem,
            plus_reduce_key=plus_reduce,
            times_reduce_key=times_reduce,
            plus_elem=ops[plus_elem],
            times_elem=ops[times_elem],
            plus_reduce=ops[plus_reduce],
            times_reduce=ops[times_reduce],
        )

    @property
    def op_keys(self) -> tuple[str, str, str, str]:
        return (
            self.plus_elem_key,
            self.times_elem_key,
            self.plus_reduce_key,
            self.times_reduce_key,
        )

@dataclass(frozen=True)
class ContractExpr(TensorExpr):
    semiring: TensorSemiring
    equation: Equation
    operands: tuple[TensorExpr, ...]

    def __post_init__(self):
        object.__setattr__(self, "operands", tuple(self.operands))

        if len(self.equation.inputs) != len(self.operands):
            raise ValueError(
                f"Equation {self.equation} expects {len(self.equation.inputs)} operands; "
                f"got {len(self.operands)}"
            )

        for i, (expected, operand) in enumerate(zip(self.equation.inputs, self.operands)):
            actual = operand.output_labels
            if actual != expected:
                raise ValueError(
                    f"Operand {i} has labels {show_labels(actual)}, "
                    f"but equation expects {show_labels(expected)} in {self.equation}"
                )

    @property
    def output_labels(self) -> Labels:
        return self.equation.output

    def __repr__(self) -> str:
        return f"contract[{self.semiring.name}]({self.equation})"

def contract(semiring: TensorSemiring, equation: str | Equation, *operands: TensorExpr) -> ContractExpr:
    if isinstance(equation, str):
        equation = Equation.parse(equation)
    return ContractExpr(semiring, equation, tuple(operands))

def show_tensor_expr(expr: TensorExpr, indent: int = 0) -> str:
    pad = " " * indent
    if isinstance(expr, TensorVar):
        return pad + repr(expr)

    if isinstance(expr, ContractExpr):
        lines = [pad + repr(expr)]
        for operand in expr.operands:
            lines.append(show_tensor_expr(operand, indent + 2))
        return "\n".join(lines)

    raise TypeError(f"unknown tensor expression: {expr!r}")

In [ ]:
# The scalar/backend pieces still come from the existing BackendOps layer.
# TensorSemiring only groups elementwise and reduction handles for contraction.

sr_add_mul = TensorSemiring.from_ops(
    "add/multiply",
    ops,
    plus_elem="add",
    times_elem="multiply",
    plus_reduce="reduce.add",
    times_reduce="reduce.multiply",
)

print("semiring keys:", sr_add_mul.op_keys)
print("plus morphism: ", sr_add_mul.plus_elem.dom(), "->", sr_add_mul.plus_elem.cod())
print("times morphism:", sr_add_mul.times_elem.dom(), "->", sr_add_mul.times_elem.cod())

In [ ]:
# ---------------------------------------------------------------------
# Build a nested contraction expression without raw Hydra application terms
# ---------------------------------------------------------------------

A = TensorVar("A", "ij")
B = TensorVar("B", "jk")
C = TensorVar("C", "kl")

nested = contract(
    sr_add_mul,
    "ik,kl->il",
    contract(sr_add_mul, "ij,jk->ik", A, B),
    C,
)

print(show_tensor_expr(nested))

In [ ]:
# ---------------------------------------------------------------------
# Rewriting over TensorExpr, not Hydra terms
# ---------------------------------------------------------------------

def fuse_contract_once(expr: TensorExpr) -> TensorExpr:
    """
    Fuse one nested contraction into its parent when:

      1. parent and child use the same TensorSemiring declaration;
      2. the child output labels match the parent input slot.

    Law obligation: the chosen operations must actually behave like a semiring.
    This code preserves the syntactic side-condition; it does not prove the laws.
    """
    if not isinstance(expr, ContractExpr):
        return expr

    outer = expr

    for pos, operand in enumerate(outer.operands):
        if not isinstance(operand, ContractExpr):
            continue

        inner = operand

        if inner.semiring != outer.semiring:
            continue

        if inner.output_labels != outer.equation.inputs[pos]:
            continue

        fused_equation = outer.equation.replace_input(pos, inner.equation.inputs)
        fused_operands = (
            outer.operands[:pos]
            + inner.operands
            + outer.operands[pos + 1:]
        )

        return contract(outer.semiring, fused_equation, *fused_operands)

    return expr

def rewrite_bottom_up(expr: TensorExpr, local_rule: Callable[[TensorExpr], TensorExpr]) -> TensorExpr:
    if isinstance(expr, ContractExpr):
        rewritten_operands = tuple(rewrite_bottom_up(o, local_rule) for o in expr.operands)
        if rewritten_operands != expr.operands:
            expr = contract(expr.semiring, expr.equation, *rewritten_operands)

    return local_rule(expr)

def normalize_contracts(expr: TensorExpr, *, max_rounds: int = 20) -> TensorExpr:
    cur = expr

    for _ in range(max_rounds):
        nxt = rewrite_bottom_up(cur, fuse_contract_once)
        if nxt == cur:
            return cur
        cur = nxt

    raise RuntimeError("contract rewrite did not converge")

In [ ]:
fused = normalize_contracts(nested)

print("nested:")
print(show_tensor_expr(nested))

print("\nfused:")
print(show_tensor_expr(fused))

assert isinstance(fused, ContractExpr)
assert str(fused.equation) == "ij,jk,kl->il"
assert fused.operands == (A, B, C)
assert fused.semiring == sr_add_mul

In [ ]:
# Different semiring declarations deliberately block fusion.

sr_max_mul = TensorSemiring.from_ops(
    "max/multiply",
    ops,
    plus_elem="maximum",
    times_elem="multiply",
    plus_reduce="reduce.maximum",
    times_reduce="reduce.multiply",
)

nested_mismatch = contract(
    sr_add_mul,
    "ik,kl->il",
    contract(sr_max_mul, "ij,jk->ik", A, B),
    C,
)

not_fused = normalize_contracts(nested_mismatch)

print(show_tensor_expr(not_fused))

assert not_fused == nested_mismatch

In [ ]:
# ---------------------------------------------------------------------
# Reference evaluator for tensor expressions
# ---------------------------------------------------------------------

_RUNTIME: dict[str, Callable] = {
    "add": np.add,
    "multiply": np.multiply,
    "maximum": np.maximum,
    "minimum": np.minimum,
    "logaddexp": np.logaddexp,
    "reduce.add": lambda x, axis: np.sum(x, axis=axis),
    "reduce.maximum": lambda x, axis: np.max(x, axis=axis),
    "reduce.minimum": lambda x, axis: np.min(x, axis=axis),
    "reduce.multiply": lambda x, axis: np.prod(x, axis=axis),
}

def _broadcast_for_equation(arr: np.ndarray, operand_labels: Labels, all_labels: Labels) -> np.ndarray:
    shape = []
    arr_axis = 0

    for label in all_labels:
        if label in operand_labels:
            shape.append(arr.shape[arr_axis])
            arr_axis += 1
        else:
            shape.append(1)

    return arr.reshape(shape)

def evaluate_tensor_expr(
    expr: TensorExpr,
    env: Mapping[str, np.ndarray],
    runtime: Mapping[str, Callable] = _RUNTIME,
) -> np.ndarray:
    if isinstance(expr, TensorVar):
        return np.asarray(env[expr.name])

    if not isinstance(expr, ContractExpr):
        raise TypeError(f"unknown tensor expression: {expr!r}")

    eq = expr.equation
    values = [evaluate_tensor_expr(o, env, runtime) for o in expr.operands]

    all_labels = tuple(dict.fromkeys(label for inp in eq.inputs for label in inp))
    reduce_axes = tuple(i for i, label in enumerate(all_labels) if label not in eq.output)

    factors = [
        _broadcast_for_equation(value, operand_labels, all_labels)
        for value, operand_labels in zip(values, eq.inputs)
    ]

    times = runtime[expr.semiring.times_elem_key]
    plus_reduce = runtime[expr.semiring.plus_reduce_key]

    acc = factors[0]
    for factor in factors[1:]:
        acc = times(acc, factor)

    if reduce_axes:
        acc = plus_reduce(acc, axis=reduce_axes)

    remaining = tuple(label for label in all_labels if label in eq.output)
    if remaining != eq.output:
        perm = [remaining.index(label) for label in eq.output]
        acc = np.asarray(acc).transpose(perm)

    return np.asarray(acc)

In [ ]:
rng = np.random.default_rng(0)

env = {
    "A": rng.random((5, 3)),  # ij
    "B": rng.random((3, 4)),  # jk
    "C": rng.random((4, 2)),  # kl
}

nested_out = evaluate_tensor_expr(nested, env)
fused_out = evaluate_tensor_expr(fused, env)

print("nested shape:", nested_out.shape)
print("fused shape: ", fused_out.shape)
print("equal:", np.allclose(nested_out, fused_out))

assert nested_out.shape == (5, 2)
assert fused_out.shape == (5, 2)
assert np.allclose(nested_out, fused_out)

In [ ]:
# A deeper chain normalizes by repeated local fusion.

A0 = TensorVar("A0", "abc")
B0 = TensorVar("B0", "cd")
C0 = TensorVar("C0", "bde")
D0 = TensorVar("D0", "ef")

nested_complex = contract(
    sr_add_mul,
    "abe,ef->abf",
    contract(
        sr_add_mul,
        "abd,bde->abe",
        contract(sr_add_mul, "abc,cd->abd", A0, B0),
        C0,
    ),
    D0,
)

fused_complex = normalize_contracts(nested_complex)

print("nested:")
print(show_tensor_expr(nested_complex))

print("\nfused:")
print(show_tensor_expr(fused_complex))

assert isinstance(fused_complex, ContractExpr)
assert str(fused_complex.equation) == "abc,cd,bde,ef->abf"
assert fused_complex.operands == (A0, B0, C0, D0)

In [ ]:
env_complex = {
    "A0": rng.random((2, 3, 4)),  # abc
    "B0": rng.random((4, 5)),     # cd
    "C0": rng.random((3, 5, 6)),  # bde
    "D0": rng.random((6, 7)),     # ef
}

nested_complex_out = evaluate_tensor_expr(nested_complex, env_complex)
fused_complex_out = evaluate_tensor_expr(fused_complex, env_complex)

print("nested shape:", nested_complex_out.shape)
print("fused shape: ", fused_complex_out.shape)
print("equal:", np.allclose(nested_complex_out, fused_complex_out))

assert nested_complex_out.shape == (2, 3, 7)
assert fused_complex_out.shape == (2, 3, 7)
assert np.allclose(nested_complex_out, fused_complex_out)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from collections.abc import Callable, Mapping
import numpy as np

from unialg import Morphism, Semiring, ProductType, identity, compose, par
from unialg.objects import ExpType, TypeUnit
from unialg.syntax import expressions as expr
from unialg.semantics import morphisms as sem

import hydra.core as core

print("using Morphism / compose / par / identity as the contraction host")

using Morphism / compose / par / identity as the contraction host


In [ ]:
# ---------------------------------------------------------------------
# Small tensor-specific metadata
# ---------------------------------------------------------------------

Labels = tuple[str, ...]


def labels(spec: str | Labels) -> Labels:
    if isinstance(spec, tuple):
        return spec
    return tuple(spec)


def show_labels(xs: Labels) -> str:
    return "".join(xs)


@dataclass(frozen=True)
class Equation:
    inputs: tuple[Labels, ...]
    output: Labels

    @classmethod
    def parse(cls, text: str) -> "Equation":
        lhs, rhs = text.split("->", 1)
        ins = tuple(labels(part.strip()) for part in lhs.split(",") if part.strip())
        return cls(ins, labels(rhs.strip()))

    def replace_input(self, pos: int, new_inputs: tuple[Labels, ...]) -> "Equation":
        return Equation(
            self.inputs[:pos] + new_inputs + self.inputs[pos + 1:],
            self.output,
        )

    def __str__(self) -> str:
        return ",".join(show_labels(i) for i in self.inputs) + "->" + show_labels(self.output)


def _idx_type(label: str):
    """Symbolic index object. Only equality matters for this notebook section."""
    return core.TypeVariable(core.Name(f"idx.{label}"))


def _index_product(ls: Labels):
    """Left-nested product of index objects."""
    if not ls:
        return TypeUnit()

    cur = _idx_type(ls[0])
    for label in ls[1:]:
        cur = ProductType(cur, _idx_type(label))
    return cur


def tensor_type(ls: str | Labels, carrier):
    """Tensor indexed by `ls`, represented as an exponential object I -> carrier."""
    return ExpType(_index_product(labels(ls)), carrier)


def product_type_left_nested(parts):
    """A, B, C -> (A × B) × C, matching the shape produced by repeated `par`."""
    parts = tuple(parts)
    if not parts:
        return TypeUnit()
    cur = parts[0]
    for part in parts[1:]:
        cur = ProductType(cur, part)
    return cur

In [ ]:
# ---------------------------------------------------------------------
# A contraction is a Morphism, not a separate expression language
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class ContractSpec:
    semiring: Semiring
    equation: Equation

    @property
    def carrier(self):
        return self.semiring.carrier

    @property
    def input_types(self):
        return tuple(tensor_type(inp, self.carrier) for inp in self.equation.inputs)

    @property
    def output_type(self):
        return tensor_type(self.equation.output, self.carrier)

    @property
    def dom(self):
        return product_type_left_nested(self.input_types)

    @property
    def cod(self):
        return self.output_type


def _semiring_aux_primitives(sr: Semiring) -> tuple:
    """Preserve backend primitives carried by semiring morphisms, if any."""
    out = []
    for m in (sr.plus, sr.times, sr.zero, sr.one):
        out.extend(getattr(m, "aux_primitives", ()))
    return tuple(out)


def contract_morphism(semiring: Semiring, equation: str | Equation) -> Morphism:
    """
    Build a typed semantic morphism for a contraction.

    The raw payload is structured metadata for rewrites. It is intentionally not
    a lowered Hydra term yet. Lowering can later interpret ContractSpec.
    """
    if isinstance(equation, str):
        equation = Equation.parse(equation)

    spec = ContractSpec(semiring, equation)
    return Morphism(
        expr.Prim(spec, spec.dom, spec.cod),
        aux_primitives=_semiring_aux_primitives(semiring),
    )


def contract_spec(node_or_morphism) -> ContractSpec | None:
    node = node_or_morphism.node if isinstance(node_or_morphism, Morphism) else node_or_morphism
    if isinstance(node, expr.Prim) and isinstance(node.raw, ContractSpec):
        return node.raw
    return None

In [ ]:
# ---------------------------------------------------------------------
# Build the nested contraction using existing semantic combinators
# ---------------------------------------------------------------------

# Uses the Semiring defined earlier in section 13:
# int_semiring = Semiring(... plus=add_pair, times=mul_pair, ...)

inner = contract_morphism(int_semiring, "ij,jk->ik")
outer = contract_morphism(int_semiring, "ik,kl->il")

T_kl = tensor_type("kl", int_semiring.carrier)

# ((T_ij × T_jk) × T_kl) -> (T_ik × T_kl) -> T_il
nested_program = compose(
    par(inner, identity(T_kl)),
    outer,
)

print("inner: ", show_morphism(inner))
print("outer: ", show_morphism(outer))
print("nested:", show_morphism(nested_program))

inner:  TypeVariable(value=Name(value='idx.i')) x TypeVariable(value=Name(value='idx.j')) -> Int x TypeVariable(value=Name(value='idx.j')) x TypeVariable(value=Name(value='idx.k')) -> Int -> TypeVariable(value=Name(value='idx.i')) x TypeVariable(value=Name(value='idx.k')) -> Int
outer:  TypeVariable(value=Name(value='idx.i')) x TypeVariable(value=Name(value='idx.k')) -> Int x TypeVariable(value=Name(value='idx.k')) x TypeVariable(value=Name(value='idx.l')) -> Int -> TypeVariable(value=Name(value='idx.i')) x TypeVariable(value=Name(value='idx.l')) -> Int
nested: param (<function ProductType at 0x7f1644339c60>, (<function ProductType at 0x7f1644339c60>, <hydra.core.TypeUnit object at 0x7f1644145f40>)), input TypeVariable(value=Name(value='idx.i')) x TypeVariable(value=Name(value='idx.j')) -> Int x TypeVariable(value=Name(value='idx.j')) x TypeVariable(value=Name(value='idx.k')) -> Int x TypeVariable(value=Name(value='idx.k')) x TypeVariable(value=Name(value='idx.l')) -> Int -> TypeVariab

In [ ]:
# ---------------------------------------------------------------------
# Reader for semantic morphism trees
# ---------------------------------------------------------------------

def show_semantic_tree(m_or_node, indent: int = 0) -> str:
    node = m_or_node.node if isinstance(m_or_node, Morphism) else m_or_node
    pad = " " * indent

    spec = contract_spec(node)
    if spec is not None:
        return pad + f"contract[{spec.semiring.name}]({spec.equation})"

    match node:
        case expr.Identity():
            return pad + "id"
        case expr.Compose(f=f, g=g):
            return "\n".join([
                pad + "compose(",
                show_semantic_tree(f, indent + 2) + ",",
                show_semantic_tree(g, indent + 2),
                pad + ")",
            ])
        case expr.Parallel(f=f, g=g):
            return "\n".join([
                pad + "par(",
                show_semantic_tree(f, indent + 2) + ",",
                show_semantic_tree(g, indent + 2),
                pad + ")",
            ])
        case expr.Pair(f=f, g=g):
            return "\n".join([
                pad + "pair(",
                show_semantic_tree(f, indent + 2) + ",",
                show_semantic_tree(g, indent + 2),
                pad + ")",
            ])
        case _:
            return pad + type(node).__name__


print(show_semantic_tree(nested_program))

compose(
  par(
    contract[int-add-mul](ij,jk->ik),
    id
  ),
  contract[int-add-mul](ik,kl->il)
)


In [ ]:
# ---------------------------------------------------------------------
# Fusion rewrite over MorphismExpr / Morphism structure
# ---------------------------------------------------------------------

def _is_identity(node) -> bool:
    return isinstance(node, expr.Identity)


def _as_plain_morphism(node) -> Morphism:
    """
    Re-wrap a semantic node as a Morphism.

    This notebook rule is for the plain tensor fragment. The real codebase can
    extend this by preserving contextual param/monad fields from ContextualBinary.
    """
    return Morphism(node)


def fuse_parallel_then_contract_once(m: Morphism) -> Morphism:
    """
    Recognize and fuse:

        compose(par(inner_contract, id), outer_contract)
        compose(par(id, inner_contract), outer_contract)

    into a single contract_morphism(...).
    """
    node = m.node
    if not isinstance(node, expr.Compose):
        return m

    if not isinstance(node.f, expr.Parallel):
        return m

    outer = contract_spec(node.g)
    if outer is None:
        return m

    left = node.f.f
    right = node.f.g

    candidates = []
    if contract_spec(left) is not None and _is_identity(right):
        candidates.append((0, contract_spec(left)))
    if _is_identity(left) and contract_spec(right) is not None:
        candidates.append((1, contract_spec(right)))

    for slot, inner in candidates:
        if inner.semiring != outer.semiring:
            continue

        if slot >= len(outer.equation.inputs):
            continue

        if inner.equation.output != outer.equation.inputs[slot]:
            continue

        fused_eq = outer.equation.replace_input(slot, inner.equation.inputs)
        fused = contract_morphism(outer.semiring, fused_eq)

        # Important: the semantics layer still guards source/target compatibility.
        if fused.dom() != m.dom() or fused.cod() != m.cod():
            raise AssertionError(
                "fusion changed visible type: "
                f"{sem.show_type(m.dom())}->{sem.show_type(m.cod())} vs "
                f"{sem.show_type(fused.dom())}->{sem.show_type(fused.cod())}"
            )

        return fused

    return m


def rewrite_children(m: Morphism) -> Morphism:
    """Bottom-up traversal, rebuilding with existing combinators."""
    node = m.node

    if isinstance(node, expr.Compose):
        f = normalize_semantic_contracts(_as_plain_morphism(node.f))
        g = normalize_semantic_contracts(_as_plain_morphism(node.g))
        return compose(f, g)

    if isinstance(node, expr.Parallel):
        f = normalize_semantic_contracts(_as_plain_morphism(node.f))
        g = normalize_semantic_contracts(_as_plain_morphism(node.g))
        return par(f, g)

    return m


def normalize_semantic_contracts(m: Morphism, *, max_rounds: int = 20) -> Morphism:
    cur = m
    for _ in range(max_rounds):
        nxt = fuse_parallel_then_contract_once(rewrite_children(cur))
        if nxt == cur:
            return cur
        cur = nxt
    raise RuntimeError("semantic contraction rewrite did not converge")

In [ ]:
fused_program = normalize_semantic_contracts(nested_program)

print("before:")
print(show_semantic_tree(nested_program))

print("\nafter:")
print(show_semantic_tree(fused_program))

fused_spec = contract_spec(fused_program)
assert fused_spec is not None
assert str(fused_spec.equation) == "ij,jk,kl->il"
assert fused_program.dom() == nested_program.dom()
assert fused_program.cod() == nested_program.cod()

before:
compose(
  par(
    contract[int-add-mul](ij,jk->ik),
    id
  ),
  contract[int-add-mul](ik,kl->il)
)

after:
contract[int-add-mul](ij,jk,kl->il)


In [ ]:
# ---------------------------------------------------------------------
# Semiring mismatch blocks fusion
# ---------------------------------------------------------------------

other_semiring = Semiring(
    name="different-int-algebra",
    carrier=INT,
    plus=mul_pair,
    times=add_pair,
    zero=const_one,
    one=const_zero,
)

inner_other = contract_morphism(other_semiring, "ij,jk->ik")

nested_mismatch = compose(
    par(inner_other, identity(T_kl)),
    outer,
)

not_fused = normalize_semantic_contracts(nested_mismatch)

print(show_semantic_tree(not_fused))
assert not_fused == nested_mismatch

compose(
  par(
    contract[different-int-algebra](ij,jk->ik),
    id
  ),
  contract[int-add-mul](ik,kl->il)
)


In [ ]:
# ---------------------------------------------------------------------
# A deeper chain also fuses bottom-up
# ---------------------------------------------------------------------

ab = contract_morphism(int_semiring, "abc,cd->abd")
abc = contract_morphism(int_semiring, "abd,bde->abe")
abcd = contract_morphism(int_semiring, "abe,ef->abf")

T_bde = tensor_type("bde", int_semiring.carrier)
T_ef = tensor_type("ef", int_semiring.carrier)

nested_complex = compose(
    par(
        compose(
            par(ab, identity(T_bde)),
            abc,
        ),
        identity(T_ef),
    ),
    abcd,
)

fused_complex = normalize_semantic_contracts(nested_complex)

print("before:")
print(show_semantic_tree(nested_complex))

print("\nafter:")
print(show_semantic_tree(fused_complex))

complex_spec = contract_spec(fused_complex)
assert complex_spec is not None
assert str(complex_spec.equation) == "abc,cd,bde,ef->abf"
assert fused_complex.dom() == nested_complex.dom()
assert fused_complex.cod() == nested_complex.cod()

before:
compose(
  par(
    compose(
      par(
        contract[int-add-mul](abc,cd->abd),
        id
      ),
      contract[int-add-mul](abd,bde->abe)
    ),
    id
  ),
  contract[int-add-mul](abe,ef->abf)
)

after:
contract[int-add-mul](abc,cd,bde,ef->abf)


In [ ]:
### Optional reference interpreter

This interpreter is only for checking the notebook examples. It interprets the semantic morphism tree directly: `Compose`, `Parallel`, `Identity`, and `ContractSpec`.

It does not lower to Hydra yet. The point is to verify that the semantic rewrite preserves behavior before adding a real lowering rule for `ContractSpec`.

In [ ]:
# ---------------------------------------------------------------------
# Reference evaluator for the semantic tensor fragment
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class RuntimeSemiring:
    times: Callable
    plus_reduce: Callable


_RUNTIME_BY_NAME: dict[str, RuntimeSemiring] = {
    "int-add-mul": RuntimeSemiring(
        times=np.multiply,
        plus_reduce=lambda x, axis: np.sum(x, axis=axis),
    ),
}


def _flatten_left_nested(value, n: int):
    if n == 1:
        return [value]
    if n == 2:
        left, right = value
        return [left, right]
    left, right = value
    return _flatten_left_nested(left, n - 1) + [right]


def _broadcast_for_equation(arr: np.ndarray, operand_labels: Labels, all_labels: Labels) -> np.ndarray:
    shape = []
    arr_axis = 0
    for label in all_labels:
        if label in operand_labels:
            shape.append(arr.shape[arr_axis])
            arr_axis += 1
        else:
            shape.append(1)
    return arr.reshape(shape)


def eval_contract_spec(spec: ContractSpec, value) -> np.ndarray:
    operands = [np.asarray(x) for x in _flatten_left_nested(value, len(spec.equation.inputs))]
    runtime = _RUNTIME_BY_NAME[spec.semiring.name]

    all_labels = tuple(dict.fromkeys(label for inp in spec.equation.inputs for label in inp))
    reduce_axes = tuple(i for i, label in enumerate(all_labels) if label not in spec.equation.output)

    factors = [
        _broadcast_for_equation(v, labs, all_labels)
        for v, labs in zip(operands, spec.equation.inputs)
    ]

    acc = factors[0]
    for factor in factors[1:]:
        acc = runtime.times(acc, factor)

    if reduce_axes:
        acc = runtime.plus_reduce(acc, axis=reduce_axes)

    remaining = tuple(label for label in all_labels if label in spec.equation.output)
    if remaining != spec.equation.output:
        perm = [remaining.index(label) for label in spec.equation.output]
        acc = np.asarray(acc).transpose(perm)

    return np.asarray(acc)


def eval_semantic_tensor_morphism(m_or_node, value):
    node = m_or_node.node if isinstance(m_or_node, Morphism) else m_or_node

    spec = contract_spec(node)
    if spec is not None:
        return eval_contract_spec(spec, value)

    match node:
        case expr.Identity():
            return value
        case expr.Compose(f=f, g=g):
            return eval_semantic_tensor_morphism(g, eval_semantic_tensor_morphism(f, value))
        case expr.Parallel(f=f, g=g):
            left, right = value
            return (
                eval_semantic_tensor_morphism(f, left),
                eval_semantic_tensor_morphism(g, right),
            )
        case _:
            raise TypeError(f"No reference evaluator for {type(node).__name__}")

In [ ]:
rng = np.random.default_rng(0)

A = rng.random((5, 3))  # ij
B = rng.random((3, 4))  # jk
C = rng.random((4, 2))  # kl

nested_arg = ((A, B), C)

nested_out = eval_semantic_tensor_morphism(nested_program, nested_arg)
fused_out = eval_semantic_tensor_morphism(fused_program, nested_arg)

print("nested shape:", nested_out.shape)
print("fused shape: ", fused_out.shape)
print("equal:", np.allclose(nested_out, fused_out))

assert nested_out.shape == (5, 2)
assert fused_out.shape == (5, 2)
assert np.allclose(nested_out, fused_out)

In [ ]:
env_complex = {
    "A0": rng.random((2, 3, 4)),  # abc
    "B0": rng.random((4, 5)),     # cd
    "C0": rng.random((3, 5, 6)),  # bde
    "D0": rng.random((6, 7)),     # ef
}

nested_complex_arg = (((env_complex["A0"], env_complex["B0"]), env_complex["C0"]), env_complex["D0"])

nested_complex_out = eval_semantic_tensor_morphism(nested_complex, nested_complex_arg)
fused_complex_out = eval_semantic_tensor_morphism(fused_complex, nested_complex_arg)

print("nested shape:", nested_complex_out.shape)
print("fused shape: ", fused_complex_out.shape)
print("equal:", np.allclose(nested_complex_out, fused_complex_out))

assert nested_complex_out.shape == (2, 3, 7)
assert fused_complex_out.shape == (2, 3, 7)
assert np.allclose(nested_complex_out, fused_complex_out)

In [ ]:
### What this section is now testing

The rewrite target is no longer raw Hydra application syntax and no longer a standalone tensor expression tree.

It is this semantic pattern:

```python
compose(
    par(contract_morphism(sr, inner_eq), identity(extra_tensor_type)),
    contract_morphism(sr, outer_eq),
)

In [ ]:
from pathlib import Path

from unialg.structure.backend import BackendOps
from unialg.tensors.semirings import Semiring
from unialg.semantics import morphisms as sem
from unialg.syntax import expressions as expr
from unialg.objects import ProductType, TypeUnit, show_type

import hydra.dsl.meta.phantoms as P

SPEC = Path("/home/scanbot/unialg/src/unialg/tensors/backends/numpy.json")
ops = BackendOps.from_spec(SPEC)

print("loaded ops:")
print(sorted(ops.keys()))

In [ ]:
import hydra.core as core

def pp_type(t):
    match t:
        case core.TypeLiteral(value=core.Name(value=name)):
            return name
        case core.TypePair(value=pt):
            return f"({pp_type(pt.first)} × {pp_type(pt.second)})"
        case core.TypeUnit():
            return "Unit"
        case core.TypeFunction(value=ft):
            return f"{pp_type(ft.domain)} -> {pp_type(ft.codomain)}"
        case _:
            return repr(t)

def pp_morphism(label, m):
    print(label)
    print("  dom:", pp_type(m.dom()))
    print("  cod:", pp_type(m.cod()))
    print("  aux:", [p.name.value for p in getattr(m, "aux_primitives", ())])

In [ ]:
from unialg.tensors.semirings import Semiring
from unialg.objects import ProductType, TypeUnit
from unialg.syntax import expressions as expr
from unialg.semantics import morphisms as sem
import hydra.dsl.meta.phantoms as P

add = ops["add"]
mul = ops["multiply"]

FLOAT = add.cod()

def const_float(value, carrier=FLOAT):
    raw = P.constant(P.float64(float(value))).value
    return sem.Morphism(expr.Prim(raw, TypeUnit(), carrier))

zero = const_float(0.0)
one = const_float(1.0)

float_add_mul = Semiring(
    name="float-add-mul",
    carrier=FLOAT,
    plus=add,
    times=mul,
    zero=zero,
    one=one,
)

pp_morphism("plus", float_add_mul.plus)
pp_morphism("times", float_add_mul.times)
pp_morphism("zero", float_add_mul.zero)
pp_morphism("one", float_add_mul.one)

In [ ]:
carrier2 = ProductType(float_add_mul.carrier, float_add_mul.carrier)

assert float_add_mul.plus.dom() == carrier2
assert float_add_mul.plus.cod() == float_add_mul.carrier

assert float_add_mul.times.dom() == carrier2
assert float_add_mul.times.cod() == float_add_mul.carrier

assert float_add_mul.zero.dom() == TypeUnit()
assert float_add_mul.zero.cod() == float_add_mul.carrier

assert float_add_mul.one.dom() == TypeUnit()
assert float_add_mul.one.cod() == float_add_mul.carrier

print("semiring shape ok")

In [ ]:
dup = sem._copy(float_add_mul.carrier)

double = sem.compose(
    dup,
    float_add_mul.plus,
)

pp_morphism("dup", dup)
pp_morphism("double = copy ; plus", double)

In [ ]:
plus_and_times = sem.par(
    float_add_mul.plus,
    float_add_mul.times,
)

pp_morphism("plus_and_times", plus_and_times)

In [ ]:
from dataclasses import dataclass

from unialg.objects import ExpType, ProductType, TypeUnit
from unialg.syntax import expressions as expr
from unialg.semantics import morphisms as sem
from unialg import Morphism
import hydra.core as core


Labels = tuple[str, ...]

def labels(s: str | Labels) -> Labels:
    return s if isinstance(s, tuple) else tuple(s)

def show_labels(xs: Labels) -> str:
    return "".join(xs)

@dataclass(frozen=True)
class Equation:
    inputs: tuple[Labels, ...]
    output: Labels

    @classmethod
    def parse(cls, text: str):
        lhs, rhs = text.split("->", 1)
        return cls(
            tuple(labels(x.strip()) for x in lhs.split(",")),
            labels(rhs.strip()),
        )

    def __str__(self):
        return ",".join(show_labels(x) for x in self.inputs) + "->" + show_labels(self.output)


def idx_type(label: str):
    return core.TypeVariable(core.Name(f"idx.{label}"))

def index_product(ls: Labels):
    if not ls:
        return TypeUnit()

    cur = idx_type(ls[0])
    for x in ls[1:]:
        cur = ProductType(cur, idx_type(x))
    return cur

def tensor_type(ls: str | Labels, carrier):
    # Tensor[I, A] ≈ I -> A
    return ExpType(index_product(labels(ls)), carrier)

def product_left(parts):
    parts = tuple(parts)
    if not parts:
        return TypeUnit()

    cur = parts[0]
    for p in parts[1:]:
        cur = ProductType(cur, p)
    return cur


@dataclass(frozen=True)
class ContractSpec:
    semiring: Semiring
    equation: Equation

    @property
    def input_types(self):
        return tuple(tensor_type(i, self.semiring.carrier) for i in self.equation.inputs)

    @property
    def dom(self):
        return product_left(self.input_types)

    @property
    def cod(self):
        return tensor_type(self.equation.output, self.semiring.carrier)


def semiring_aux(sr: Semiring):
    out = []
    for m in (sr.plus, sr.times, sr.zero, sr.one):
        out.extend(getattr(m, "aux_primitives", ()))
    return tuple(out)


def contract_morphism(sr: Semiring, equation: str | Equation) -> Morphism:
    if isinstance(equation, str):
        equation = Equation.parse(equation)

    spec = ContractSpec(sr, equation)

    return Morphism(
        expr.Prim(spec, spec.dom, spec.cod),
        aux_primitives=semiring_aux(sr),
    )


def contract_spec(m_or_node):
    node = m_or_node.node if isinstance(m_or_node, Morphism) else m_or_node
    if isinstance(node, expr.Prim) and isinstance(node.raw, ContractSpec):
        return node.raw
    return None

In [ ]:
inner = contract_morphism(float_add_mul, "ij,jk->ik")
outer = contract_morphism(float_add_mul, "ik,kl->il")

T_kl = tensor_type("kl", float_add_mul.carrier)

nested = sem.compose(
    sem.par(inner, sem._identity(T_kl)),
    outer,
)

pp_morphism("inner", inner)
pp_morphism("outer", outer)
pp_morphism("nested", nested)

print("inner contract:", contract_spec(inner).equation)
print("outer contract:", contract_spec(outer).equation)
print("nested aux:", [p.name.value for p in nested.aux_primitives])

In [ ]:
def as_morphism(node_or_m):
    return node_or_m if isinstance(node_or_m, Morphism) else Morphism(node_or_m)


def is_identity_node(node) -> bool:
    return isinstance(node, expr.Identity)


def flatten_par_tree(node) -> list:
    """
    Flatten a binary Parallel tree into semantic leaves.

    par(par(a, b), c) -> [a, b, c]
    par(a, par(b, c)) -> [a, b, c]

    Product association is still represented by the actual Morphism types;
    this helper is only for recognizing the routing pattern.
    """
    if isinstance(node, expr.Parallel):
        return flatten_par_tree(node.f) + flatten_par_tree(node.g)
    return [node]


def fuse_contract_inputs_once(m: Morphism) -> Morphism:
    node = m.node

    if not isinstance(node, expr.Compose):
        return m

    outer = contract_spec(node.g)
    if outer is None:
        return m

    route_leaves = flatten_par_tree(node.f)

    if len(route_leaves) != len(outer.equation.inputs):
        return m

    new_inputs = []

    for slot, leaf in enumerate(route_leaves):
        outer_input = outer.equation.inputs[slot]
        inner = contract_spec(leaf)

        if inner is None:
            if is_identity_node(leaf):
                # This slot is untouched.
                new_inputs.append(outer_input)
                continue

            # Some nontrivial morphism feeds this slot.
            # Do not fuse unless we know how to absorb it.
            return m

        if inner.semiring != outer.semiring:
            return m

        if inner.equation.output != outer_input:
            return m

        # Replace this outer input by all inputs of the inner contraction.
        new_inputs.extend(inner.equation.inputs)

    fused_eq = Equation(tuple(new_inputs), outer.equation.output)
    fused = contract_morphism(outer.semiring, fused_eq)

    # Let the semantic layer guard visible type preservation.
    if fused.dom() != m.dom() or fused.cod() != m.cod():
        return m

    return fused

In [ ]:
def normalize_contracts(m: Morphism, *, max_rounds: int = 100) -> Morphism:
    def rebuild_children(x: Morphism) -> Morphism:
        node = x.node

        if isinstance(node, expr.Compose):
            f = normalize_contracts(as_morphism(node.f), max_rounds=max_rounds)
            g = normalize_contracts(as_morphism(node.g), max_rounds=max_rounds)
            return sem.compose(f, g)

        if isinstance(node, expr.Parallel):
            f = normalize_contracts(as_morphism(node.f), max_rounds=max_rounds)
            g = normalize_contracts(as_morphism(node.g), max_rounds=max_rounds)
            return sem.par(f, g)

        if isinstance(node, expr.Pair):
            f = normalize_contracts(as_morphism(node.f), max_rounds=max_rounds)
            g = normalize_contracts(as_morphism(node.g), max_rounds=max_rounds)
            return sem.pair(f, g)

        return x

    cur = m

    for _ in range(max_rounds):
        nxt = fuse_contract_inputs_once(rebuild_children(cur))
        if nxt == cur:
            return cur
        cur = nxt

    raise RuntimeError("contract normalization did not converge")

In [ ]:
inner/outer is not the model.

A contract feeding a contract is the local rewrite.
Arbitrary nested contraction programs are handled by:
    1. semantic tree traversal
    2. local fusion
    3. repeat to fixed point

compose
par
pair
identity
domain/codomain compatibility
aux primitive propagation

Equation
ContractSpec
contract_morphism(...)
fusion rule over ContractSpec metadata

In [ ]:
The useful parts of `dsl-guide-python.md` for your current `unialg` work are more specific than “Hydra has DSLs.”

## 1. Use Hydra DSLs at the realization/backend boundary, not as the semantic language

The guide separates Hydra-Python into these layers:

```text
Direct DSLs:
    hydra.dsl.types
    hydra.dsl.terms

Phantom-typed DSL:
    hydra.dsl.meta.phantoms

Domain DSLs:
    hydra.dsl.meta.core / graph / compute

Library wrappers:
    hydra.dsl.meta.lib.*
```

It explicitly says direct DSLs are for raw `Type`/`Term` construction, while phantom/domain DSLs are for writing Hydra kernel-style source terms. ([GitHub][1])

For `unialg`, that means:

```text
unialg.syntax / semantics:
    keep your own MorphismExpr / Morphism / Functor / Optic language

unialg.structure.realize:
    use P.*, Core.*, library wrappers to emit Hydra terms

unialg.tensors.math / BackendOps:
    use Hydra primitives as executable backend leaves
```

So the right direction is still:

```text
semantic rewrite over MorphismExpr
    then lower to Hydra terms
```

not raw Hydra term rewriting as the primary tensor optimizer.

## 2. Your backend `"float"` type is probably wrong

The guide lists concrete float types as:

```python
T.float32()
T.float64()
```

There is no generic `T.float()` in the direct type DSL examples. ([GitHub][1])

That explains your earlier failure:

```python
TypeLiteral(value=Name(value='float'))
```

Hydra’s own type display expected one of the real literal variants, not a generic `Name("float")`.

So the backend spec loader should probably map backend scalar aliases explicitly:

```python
import hydra.dsl.types as T

BACKEND_TYPE_ALIASES = {
    "float": T.float64(),     # choose deliberately
    "float64": T.float64(),
    "float32": T.float32(),
    "int32": T.int32(),
    "boolean": T.boolean(),
}
```

Then `ops["add"].dom()` would become a canonical Hydra type, and `show_type` should stop exploding.

The current state works structurally because equality still sees the same fake type, but it is not aligned with Hydra’s canonical type layer.

## 3. BackendOps is aligned with Hydra primitive usage

The guide shows primitive references and primitive calls through:

```python
P.primitive(Name(...))
P.primitive1(...)
P.primitive2(...)
```

and the examples treat primitive wrappers as the correct way to call Hydra-hosted functions. ([GitHub][1])

That matches what you observed:

```text
ops["add"]
  = Morphism(
      node=Prim(lambda x. unialg.backend.add(first x)(second x)),
      dom=float × float,
      cod=float,
      aux_primitives=(unialg.backend.add,)
    )
```

So the backend path is correct:

```text
backend JSON
    -> registered Hydra Primitive
    -> pair-taking Morphism
    -> Semiring.plus / Semiring.times
    -> compose / par / pair
```

The main fix is the type parser, not the overall architecture.

## 4. `P.compose` order is a footgun

The guide shows:

```python
composed = P.compose(P.var("g"), P.var("f"))
```

That is the usual function composition shape: `g ∘ f`. ([GitHub][1])

Your `unialg.compose(f, g)` is intentionally diagrammatic:

```text
f first, then g
```

So in `realize.py`, whenever `MorphismExpr.Compose(f, g)` lowers to Hydra, it likely needs to emit:

```python
P.compose(realize(g), realize(f))
```

not:

```python
P.compose(realize(f), realize(g))
```

This is one of the places where keeping `unialg` semantics separate from Hydra syntax matters.

## 5. Recursive morphisms should use qualified self references

The guide explicitly says Python module-level recursive values cannot self-reference during construction, and recommends qualified references like:

```python
P.var("my.namespace.functionName")
```

for recursive definitions. ([GitHub][1])

That supports your current `cata` / `ana` / `hylo` direction: recursive carriers should generate a named self-reference and then add the actual body into the graph/aux primitive context later.

So this pattern is good:

```text
recursive Morphism
    contains TermVariable("unialg.generated.some_rec")
    plus aux graph/primitives binding that name
```

not:

```text
Python object directly closes over itself during construction
```

## 6. Hydra rewriting examples are useful, but at the lower layer

The doc points to `hydra.rewriting` examples covering recursive rewriting, structural rewriting, folds, and binding-aware rewriting. ([GitHub][1])

That is useful for **Hydra term cleanup**, but tensor contraction fusion should first operate over your semantic tree:

```text
Compose
Parallel
Pair
Identity
Prim(ContractSpec)
```

because fan-out, sharing, product routing, and contraction metadata are still visible there.

Once lowered to Hydra `TermApplication`, you have already lost much of the semantic structure that tells you whether a contraction is single-use, shared, duplicated, or materialized.

## Concrete change I would make now

Fix backend type loading before building more tensor machinery.

The current successful output:

```text
plus
  dom: (float × float)
  cod: float
```

is semantically promising but Hydra-type-canonicality is off.

I would update the backend spec parser so `"float"` becomes either `T.float64()` or a deliberate backend scalar alias object. For the notebook, use `float64` unless you specifically need backend-polymorphic scalar types.

Then the path becomes clean:

```text
BackendOps.from_spec(...)
    produces canonical Hydra types

Semiring(...)
    stores canonical Morphism operations

contract_morphism(sr, eq)
    can use sr.carrier safely

rewrite
    operates over semantic Morphism structure

realize
    emits Hydra terms using P.* DSL conventions
```

[1]: https://github.com/CategoricalData/hydra/blob/main/docs/dsl-guide-python.md "hydra/docs/dsl-guide-python.md at main · CategoricalData/hydra · GitHub"


In [ ]:
# ---------------------------------------------------------------------
# Notebook syntax sugar: sections + pipelines over existing Morphisms
# ---------------------------------------------------------------------
#
# This adds only surface syntax.
# It desugars into existing semantic combinators:
#
#   Morphism
#   compose
#   pair
#   par
#   _identity
#
# Examples:
#
#   add[_, 1.0]              x ↦ x + 1
#   mul[2.0, _]              x ↦ 2 * x
#   add[_, 1.0] >> mul[2.0, _]
#                             x ↦ 2 * (x + 1)
#
#   f & g                   pair(f, g), same input fanout
#   f | g                   par(f, g), product-side map

from __future__ import annotations

from dataclasses import dataclass
from functools import reduce

import hydra.dsl.meta.phantoms as P

from unialg.semantics.morphisms import _identity, pair, par, compose, Morphism
from unialg.syntax import expressions as expr


# ---------------------------------------------------------------------
# Placeholder
# ---------------------------------------------------------------------

class Hole:
    def __repr__(self):
        return "_"

_ = Hole()


def _unwrap(m):
    return m.m if isinstance(m, Arrow) else m


def _product_parts(t):
    return t.value.first, t.value.second


# ---------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------

def const_float(value: float, cod, *, dom=None) -> Morphism:
    """
    Constant morphism.

    If dom is provided:
        dom -> cod

    If dom is omitted:
        cod -> cod

    This uses P.constant directly. Do not build a Hydra Lambda here.
    """
    if dom is None:
        dom = cod

    raw = P.constant(P.float64(float(value))).value
    return Morphism(expr.Prim(raw, dom, cod))


# ---------------------------------------------------------------------
# Sectioning
# ---------------------------------------------------------------------

def section(op: Morphism, const, value, *, side: str = "right") -> Morphism:
    """
    Section a binary morphism.

        op : A × B -> C

    side="right":
        section(op, c, b) : A -> C
        x ↦ op(x, b)

    side="left":
        section(op, c, a) : B -> C
        x ↦ op(a, x)
    """
    left_t, right_t = _product_parts(op.dom())
    types = [left_t, right_t]

    fixed = {"left": 0, "right": 1}[side]
    open_ = 1 - fixed

    open_type = types[open_]
    fixed_type = types[fixed]

    parts = [None, None]
    parts[open_] = _identity(open_type)
    parts[fixed] = const(value, fixed_type, dom=open_type)

    return compose(pair(*parts), op)


# ---------------------------------------------------------------------
# Arrow wrapper
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class Arrow:
    m: Morphism
    name: str | None = None

    def __rshift__(self, other):
        """f >> g means f first, then g."""
        other = A(other)
        return Arrow(compose(self.m, other.m), f"({self} >> {other})")

    def __and__(self, other):
        """f & g means pair(f, g), same input fanout."""
        other = A(other)
        return Arrow(pair(self.m, other.m), f"({self} & {other})")

    def __or__(self, other):
        """f | g means par(f, g), product-side map."""
        other = A(other)
        return Arrow(par(self.m, other.m), f"({self} | {other})")

    def dom(self):
        return self.m.dom()

    def cod(self):
        return self.m.cod()

    def __repr__(self):
        return self.name or repr(self.m)


def A(m, name=None) -> Arrow:
    if isinstance(m, Arrow):
        return m
    return Arrow(m, name)


def pipe(*ms) -> Morphism:
    """pipe(f, g, h) = f ; g ; h."""
    return reduce(compose, [_unwrap(m) for m in ms])


# ---------------------------------------------------------------------
# Binary operation sugar
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class BinOp:
    op: Morphism
    name: str
    const: callable = const_float

    def left(self, value) -> Arrow:
        return Arrow(
            section(self.op, self.const, value, side="left"),
            f"{self.name}[{value}, _]",
        )

    def right(self, value) -> Arrow:
        return Arrow(
            section(self.op, self.const, value, side="right"),
            f"{self.name}[_, {value}]",
        )

    def __getitem__(self, args) -> Arrow:
        """
        Placeholder section syntax.

            add[_, 1.0]      x ↦ x + 1
            mul[2.0, _]      x ↦ 2 * x
        """
        if not isinstance(args, tuple) or len(args) != 2:
            raise TypeError("binary section syntax expects exactly two arguments")

        left, right = args

        if isinstance(left, Hole) and not isinstance(right, Hole):
            return self.right(right)

        if isinstance(right, Hole) and not isinstance(left, Hole):
            return self.left(left)

        raise TypeError("exactly one side must be the placeholder _")


def binop(name: str, *, const=const_float) -> BinOp:
    return BinOp(ops[name], name=name, const=const)


# ---------------------------------------------------------------------
# Example
# ---------------------------------------------------------------------

add = binop("add")
mul = binop("multiply")

add1 = add[_, 1.0]              # x ↦ x + 1
mul2 = mul[2.0, _]              # x ↦ 2 * x

add_then_double = add1 >> mul2  # x ↦ 2 * (x + 1)

print("add1:           ", add1)
print("mul2:           ", mul2)
print("add_then_double:", add_then_double)

pp_morphism("add1", add1.m)
pp_morphism("mul2", mul2.m)
pp_morphism("add_then_double", add_then_double.m)

In [ ]:
add2 = add[_, 3.0]
add_then_double_then_add = add_then_double | add2

In [ ]:
from unialg import run
import hydra.dsl.meta.phantoms as P


def float_val(term):
    # Current Hydra float literal nesting in this notebook.
    return term.value.value.value


def run_float(m_or_arrow, x: float):
    m = m_or_arrow.m if isinstance(m_or_arrow, Arrow) else m_or_arrow
    arg = P.float64(float(x)).value
    return float_val(run(m, arg, ctx, graph))


print(run_float(add_then_double_then_add, 4.0))

In [25]:
import importlib
import inspect
import textwrap

modules = [
    "checking",
    "unification",
    "inference",
    "predicates",
    "substitution",
    "rewriting",
    "analysis",
    "hoisting",
    "typing",
    "adapt",
]

In [27]:
for name in modules:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n\n########## hydra.{name} ##########")
    print("MODULE DOC:")
    print(inspect.getdoc(mod) or "<no module docstring>")

    for attr_name in sorted(n for n in dir(mod) if not n.startswith("_")):
        attr = getattr(mod, attr_name)
        doc = inspect.getdoc(attr)
        if doc:
            print(f"\n--- {attr_name} ---")
            print(textwrap.shorten(doc, width=400, placeholder=" ..."))



########## hydra.checking ##########
MODULE DOC:
Type checking and type reconstruction (type-of) for the results of Hydra unification and inference.

--- Either ---
A type alias for sum types (Left[L] | Right[R]). Subscriptable at runtime for cast() compatibility.

--- FrozenDict ---
An immutable variant of the Python dict.

--- Just ---
Just(value: 'T')

--- Left ---
Left(value: 'L')

--- Maybe ---
A type alias for optional values (Just[T] | Nothing). Subscriptable at runtime for cast() compatibility.

--- Right ---
Right(value: 'R')

--- T0 ---
Type variable. The preferred way to construct a type variable is via the dedicated syntax for generic functions, classes, and type aliases:: class Sequence[T]: # T is a TypeVar ... This syntax can also be used to create bound and constrained type variables:: # S is a TypeVar bound to str class StrSequence[S: str]: ... # A is a TypeVar constrained to str or bytes class StrOrBytesSequence[A: ...

--- T1 ---
Type variable. The preferred way to 

In [28]:
for name in modules:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n\n########## hydra.{name} ##########")

    for attr_name in sorted(n for n in dir(mod) if not n.startswith("_")):
        attr = getattr(mod, attr_name)
        try:
            sig = str(inspect.signature(attr))
        except Exception:
            sig = "<no signature>"
        doc = inspect.getdoc(attr)
        if doc or sig != "<no signature>":
            print(f"\n--- {attr_name}{sig} ---")
            if doc:
                print(textwrap.shorten(doc, width=400, placeholder=" ..."))



########## hydra.checking ##########

--- Callable() ---

--- Either() ---
A type alias for sum types (Left[L] | Right[R]). Subscriptable at runtime for cast() compatibility.

--- FrozenDict(data: 'Any' = None, *, _trusted: 'bool' = False) -> 'None' ---
An immutable variant of the Python dict.

--- Just(value: 'T') -> None ---
Just(value: 'T')

--- Left(value: 'L') -> None ---
Left(value: 'L')

--- Maybe() ---
A type alias for optional values (Just[T] | Nothing). Subscriptable at runtime for cast() compatibility.

--- Nothing() ---

--- Right(value: 'R') -> None ---
Right(value: 'R')

--- T0<no signature> ---
Type variable. The preferred way to construct a type variable is via the dedicated syntax for generic functions, classes, and type aliases:: class Sequence[T]: # T is a TypeVar ... This syntax can also be used to create bound and constrained type variables:: # S is a TypeVar bound to str class StrSequence[S: str]: ... # A is a TypeVar constrained to str or bytes class StrOrBytes

In [29]:
import importlib
import inspect

mods = [
    "serialization",
    "parsing",
    "parsers",
    "encoding",
    "decoding",
    "reflect",
    "templates",
    "sources",
    "graph",
    "coders",
    "packaging",
    "tools",
]

In [30]:
for name in mods:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n########## hydra.{name} ##########")
    for attr in sorted(n for n in dir(mod) if not n.startswith("_")):
        print(attr)


########## hydra.serialization ##########
Callable
Just
Maybe
Nothing
angle_braces
angle_braces_list
annotations
braces_list_adaptive
bracket_list
bracket_list_adaptive
brackets
cast
choose_layout
comma_sep
comma_sep_adaptive
cst
curly_block
curly_braces
curly_braces_list
custom_indent
custom_indent_block
dot_sep
double_newline_sep
double_space
expression_length
frozenlist
full_block_style
half_block_style
hydra
ifx
indent
indent_block
indent_subsequent_lines
infix_ws
infix_ws_list
inline_style
lru_cache
max_line_width
newline_sep
no_padding
no_sep
num
op
or_op
or_sep
paren_list
paren_list_adaptive
parens
parentheses
parenthesize
prefix
print_expr
semicolon_sep
sep
space_sep
space_sep_adaptive
square_brackets
structural_sep
structural_space_sep
suffix
sym
symbol_sep
tab_indent
tab_indent_double_space
tab_indent_single_space
unsupported_type
unsupported_variant
with_comma
with_semi

########## hydra.parsing ##########
A
Annotated
Callable
Generic
Node
ParseError
ParseResult
ParseResult

In [31]:
import textwrap

for name in mods:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n\n########## hydra.{name} ##########")
    for attr_name in sorted(n for n in dir(mod) if not n.startswith("_")):
        attr = getattr(mod, attr_name)
        try:
            sig = str(inspect.signature(attr))
        except Exception:
            sig = "<no signature>"
        doc = inspect.getdoc(attr)
        if doc or sig != "<no signature>":
            print(f"\n--- {attr_name}{sig} ---")
            if doc:
                print(textwrap.shorten(doc, width=500, placeholder=" ..."))



########## hydra.serialization ##########

--- Callable() ---

--- Just(value: 'T') -> None ---
Just(value: 'T')

--- Maybe() ---
A type alias for optional values (Just[T] | Nothing). Subscriptable at runtime for cast() compatibility.

--- Nothing() ---

--- angle_braces<no signature> ---
Matching open and close bracket symbols.

--- angle_braces_list(style: 'hydra.ast.BlockStyle', els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Comma-separate the elements inside angle brackets in the given block style; renders as `<>` when empty.

--- braces_list_adaptive(els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Produce a bracketed list which separates elements by spaces or newlines depending on the estimated width of the expression.

--- bracket_list(style: 'hydra.ast.BlockStyle', els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Comma-separate the elements inside square brackets in the given block style; renders as `[]` when empty.

--- bracket_list_adaptive(e

In [33]:
keywords = [
    "json", "yaml", "parse", "parser", "encode", "decode", "serialize",
    "deserialize", "module", "definition", "record", "schema", "graph",
    "source", "file", "package", "template", "coder"
]

for name in mods:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n########## hydra.{name} ##########")
    for attr_name in sorted(n for n in dir(mod) if not n.startswith("_")):
        low = attr_name.lower()
        if any(k in low for k in keywords):
            attr = getattr(mod, attr_name)
            try:
                sig = str(inspect.signature(attr))
            except Exception:
                sig = "<no signature>"
            print(f"{attr_name}{sig}")


########## hydra.serialization ##########

########## hydra.parsing ##########
ParseError(message: "Annotated[str, 'An error message']", remainder: "Annotated[str, 'The remaining input at the point of failure']") -> None
ParseResult()
ParseResultFailure(value: 'T') -> None
ParseResultSuccess(value: 'T') -> None
ParseSuccess(value: "Annotated[A, 'The parsed value']", remainder: "Annotated[str, 'The remaining unparsed input']") -> None
Parser(value: 'T') -> None

########## hydra.parsers ##########
run_parser(p: 'hydra.parsing.Parser[T0]', input: 'str') -> 'hydra.parsing.ParseResult[T0]'

########## hydra.encoding ##########
encode_binding(cx: 'T0', graph: 'hydra.graph.Graph', b: 'hydra.core.Binding') -> 'Either[hydra.errors.DecodingError, hydra.core.Binding]'
encode_binding_name(n: 'hydra.core.Name') -> 'hydra.core.Name'
encode_either_type(et: 'hydra.core.EitherType') -> 'hydra.core.Term'
encode_field_value(type_name: 'hydra.core.Name', field_name: 'hydra.core.Name', field_type: 'hydra

In [34]:
target_mods = ["serialization", "parsing", "parsers", "encoding", "decoding", "reflect", "sources"]

for name in target_mods:
    mod = importlib.import_module(f"hydra.{name}")
    print(f"\n\n########## hydra.{name} ##########")
    for attr_name in sorted(n for n in dir(mod) if not n.startswith("_")):
        attr = getattr(mod, attr_name)
        doc = inspect.getdoc(attr)
        try:
            sig = str(inspect.signature(attr))
        except Exception:
            sig = "<no signature>"
        if doc:
            print(f"\n--- {attr_name}{sig} ---")
            print(doc)



########## hydra.serialization ##########

--- Just(value: 'T') -> None ---
Just(value: 'T')

--- Maybe() ---
A type alias for optional values (Just[T] | Nothing). Subscriptable at runtime for cast() compatibility.

--- angle_braces<no signature> ---
Matching open and close bracket symbols.

--- angle_braces_list(style: 'hydra.ast.BlockStyle', els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Comma-separate the elements inside angle brackets in the given block style; renders as `<>` when empty.

--- braces_list_adaptive(els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Produce a bracketed list which separates elements by spaces or newlines depending on the estimated width of the expression.

--- bracket_list(style: 'hydra.ast.BlockStyle', els: 'frozenlist[hydra.ast.Expr]') -> 'hydra.ast.Expr' ---
Comma-separate the elements inside square brackets in the given block style; renders as `[]` when empty.

--- bracket_list_adaptive(els: 'frozenlist[hydra.ast.Expr]') -> 'h

In [35]:
import hydra.packaging as packaging
import inspect

for name in ["Library", "Module", "Package", "TermDefinition", "TypeDefinition", "Namespace", "QualifiedName"]:
    obj = getattr(packaging, name)
    print(f"\n--- {name} ---")
    print(inspect.signature(obj))
    print(inspect.getdoc(obj))


--- Library ---
(namespace: "Annotated[Namespace, 'A common prefix for all primitive function names in the library']", prefix: "Annotated[str, 'A preferred namespace prefix for function names in the library']", primitives: "Annotated[frozenlist[hydra.graph.Primitive], 'The primitives defined in this library']") -> None
A library of primitive functions.

--- Module ---
(description: "Annotated[Maybe[str], 'An optional human-readable description of the module']", namespace: "Annotated[Namespace, 'A common prefix for all element names in the module']", dependencies: "Annotated[frozenlist[Namespace], 'Any modules which this module directly depends on']", definitions: "Annotated[frozenlist[Definition], 'The definitions in this module']") -> None
A logical collection of elements in the same namespace, having dependencies on zero or more other modules.

--- Package ---
(name: "Annotated[PackageName, 'The name of the package']", modules: "Annotated[frozenlist[Module], 'The modules in this pac

In [39]:
import hydra.tools as tools

print(type(tools.TYPE_CODERS))
print(tools.TYPE_CODERS)

for k, v in tools.TYPE_CODERS.items():
    print("\nKEY:", k)
    print("VALUE:", v)

<class 'dict'>
{<class 'str'>: <function string at 0x7f34801a9120>, <class 'int'>: <function int32 at 0x7f34801a8d60>}

KEY: <class 'str'>
VALUE: <function string at 0x7f34801a9120>

KEY: <class 'int'>
VALUE: <function int32 at 0x7f34801a8d60>


In [42]:
import hydra.tools as tools

for k in tools.TYPE_CODERS.keys():
    print(repr(k), type(k))

<class 'str'> <class 'type'>
<class 'int'> <class 'type'>


In [37]:
import hydra.coders as coders
import inspect

for name in ["Coder", "Adapter", "Language", "LanguageConstraints", "AdapterContext"]:
    obj = getattr(coders, name)
    print(f"\n--- {name} ---")
    print(inspect.signature(obj))
    print(inspect.getdoc(obj))


--- Coder ---
(encode: "Annotated[Callable[[hydra.context.Context, V1], Either[hydra.errors.Error, V2]], 'A function which encodes source values as target values in a given context']", decode: "Annotated[Callable[[hydra.context.Context, V2], Either[hydra.errors.Error, V1]], 'A function which decodes target values as source values in a given context']") -> None
An encoder and decoder; a bidirectional transformation between two types.

--- Adapter ---
(is_lossy: "Annotated[bool, 'Whether information may be lost in the course of this adaptation']", source: "Annotated[T1, 'The source type']", target: "Annotated[T2, 'The target type']", coder: "Annotated[Coder[V1, V2], 'The coder for transforming instances of the source type to instances of the target type']") -> None
A two-level bidirectional encoder which adapts types to types and terms to terms.

--- Language ---
(name: "Annotated[LanguageName, 'The unique name of the language']", constraints: "Annotated[LanguageConstraints, 'The constr